# Part 3 : Snowflake Marketplace × EC 売上分析

## このノートブックでやること
1. Snowflake Marketplace から **Weather Source** 気象データを取得
2. 気象データと GlacierStyle EC 売上データを統合
3. 「天気と購買行動」の関係を分析

## ゼロコピーとは？
Marketplace から取得したデータは自分のアカウントに**コピーされない**。  
提供元のデータをそのまま参照するため：
- ストレージコストがかからない
- 常に最新のデータを利用できる
- 契約してすぐ使える

---
## 事前準備：セッション設定

In [ ]:
USE ROLE ACCOUNTADMIN;
USE DATABASE AI_HANDSON_DB;
USE SCHEMA ANALYTICS;
USE WAREHOUSE AI_HANDSON_WH;

---
## 1. Snowflake Marketplace から気象データを取得

### 手順（UI 操作）

1. 左メニューの **「Data Products」>「Marketplace」** をクリック
2. 検索バーに **「Weather Source LLC: frostbyte」** と入力
3. リストを選択し **「Get」** をクリック
4. データベース名を **`FROSTBYTE_WEATHERSOURCE`** に設定して取得

> **ポイント:** このリストは無料で取得できます。  
> データはコピーされず、Weather Source 側のストレージを参照します（ゼロコピー）。

![Marketplace Get](images/part3/01_marketplace_get.png)

### 取得できたか確認

In [ ]:
-- Weather Source データベースの確認
SHOW DATABASES LIKE 'FROSTBYTE%';

In [ ]:
-- 利用可能なテーブルの確認
SHOW TABLES IN SCHEMA FROSTBYTE_WEATHERSOURCE.ONPOINT_ID;

In [ ]:
-- 日次気象データのサンプル確認（東京）
SELECT
    date_valid_std                                     AS weather_date,
    city_name,
    ROUND(avg_temperature_air_2m_f, 1)                 AS avg_temp_f,
    ROUND((avg_temperature_air_2m_f - 32) * 5/9, 1)   AS avg_temp_c,
    tot_precipitation_in                               AS precipitation_in
FROM FROSTBYTE_WEATHERSOURCE.ONPOINT_ID.HISTORY_DAY
WHERE country   = 'JP'
  AND city_name = 'Tokyo'
ORDER BY weather_date DESC
LIMIT 20;

---
## 2. 気象データと EC 売上データの統合

GlacierStyle の注文データ（`fact_orders`）と気象データを日付で結合します。

```
fact_orders（自社データ）          HISTORY_DAY（Marketplace）
    日付・注文数・売上金額    ×      日付・気温・降水量
         ↓                               ↓
         ←── JOIN on 日付 ──────────────→
              天気と購買行動の分析
```

In [ ]:
-- 注文データの日別集計（結合前の確認）
SELECT
    DATE(order_date)    AS order_day,
    COUNT(order_id)     AS order_count,
    SUM(total_amount)   AS daily_revenue
FROM fact_orders
GROUP BY 1
ORDER BY 1 DESC
LIMIT 20;

### 日次売上 × 気象ビューを作成

In [ ]:
CREATE OR REPLACE VIEW daily_sales_weather_v AS
SELECT
    o.order_day,
    o.order_count,
    o.daily_revenue,
    ROUND(w.avg_temperature_air_2m_f, 1)               AS avg_temp_f,
    ROUND((w.avg_temperature_air_2m_f - 32) * 5/9, 1)  AS avg_temp_c,
    w.tot_precipitation_in                             AS precipitation_in,
    CASE
        WHEN w.tot_precipitation_in > 0.1 THEN '雨の日'
        ELSE '晴れ・曇り'
    END AS weather_category
FROM (
    SELECT
        DATE(order_date)   AS order_day,
        COUNT(order_id)    AS order_count,
        SUM(total_amount)  AS daily_revenue
    FROM fact_orders
    GROUP BY 1
) o
LEFT JOIN FROSTBYTE_WEATHERSOURCE.ONPOINT_ID.HISTORY_DAY w
    ON  o.order_day = w.date_valid_std
    AND w.country   = 'JP'
    AND w.city_name = 'Tokyo';

In [ ]:
-- ビューの確認
SELECT *
FROM daily_sales_weather_v
ORDER BY order_day DESC
LIMIT 20;

---
## 3. 天気と購買行動の分析

### 気温帯別の注文数・売上

In [ ]:
SELECT
    CASE
        WHEN avg_temp_c < 10 THEN '① 寒い（10℃未満）'
        WHEN avg_temp_c < 20 THEN '② 涼しい（10〜20℃）'
        WHEN avg_temp_c < 28 THEN '③ 快適（20〜28℃）'
        ELSE                      '④ 暑い（28℃以上）'
    END                          AS temp_range,
    COUNT(*)                     AS day_count,
    ROUND(AVG(order_count), 1)   AS avg_daily_orders,
    ROUND(AVG(daily_revenue), 0) AS avg_daily_revenue
FROM daily_sales_weather_v
WHERE avg_temp_c IS NOT NULL
GROUP BY 1
ORDER BY 1;

### 雨の日 vs 晴れの日の比較

In [ ]:
SELECT
    weather_category,
    COUNT(*)                     AS day_count,
    ROUND(AVG(order_count), 1)   AS avg_daily_orders,
    ROUND(AVG(daily_revenue), 0) AS avg_daily_revenue,
    ROUND(SUM(daily_revenue), 0) AS total_revenue
FROM daily_sales_weather_v
WHERE weather_category IS NOT NULL
GROUP BY 1
ORDER BY avg_daily_revenue DESC;

### 月別の気温と売上トレンド

In [ ]:
SELECT
    DATE_TRUNC('month', order_day)        AS month,
    ROUND(AVG(avg_temp_c), 1)             AS avg_temp_c,
    SUM(order_count)                      AS total_orders,
    ROUND(SUM(daily_revenue), 0)          AS total_revenue
FROM daily_sales_weather_v
WHERE avg_temp_c IS NOT NULL
GROUP BY 1
ORDER BY 1;

---
## まとめ

| 機能 | ポイント |
|---|---|
| **Snowflake Marketplace** | 外部データをゼロコピーで即利用 |
| **ゼロコピー** | ストレージコスト不要・常に最新データ |
| **データ統合** | 自社データ × 外部データで新しいインサイト発見 |

### Marketplace で使えるデータの例
- **気象データ**（Weather Source） — 天気と購買の相関
- **人口統計データ** — エリアごとのターゲット分析
- **経済指標データ** — 景気動向と売上の関係
- **POI（関心地点）データ** — 競合店舗や施設との位置関係

> 自社データだけでは気づけなかったインサイトを、外部データとの組み合わせで発見できるのが Snowflake Marketplace の強みです。

---
## クリーンアップ（任意）

In [ ]:
-- 作成したビューを削除する場合
DROP VIEW IF EXISTS AI_HANDSON_DB.ANALYTICS.daily_sales_weather_v;